# 广义矩估计 (GMM) 完整教程：从矩条件到因子模型检验

## 📚 教学目标
1. 理解**矩条件 (Moment Conditions)** 的含义与直觉
2. 掌握 **GMM 估计**的完整计算步骤：目标函数、权重矩阵、参数求解
3. 区分**一阶段 GMM** 与**两阶段 GMM**，理解效率差异
4. 理解 **J 检验（过度识别检验）** 及其在模型诊断中的作用
5. 将 GMM 应用于**因子模型**的风险溢价估计与模型检验

**参考书**：《因子投资：方法与实践》（石川）第 2.7 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到因子模型的真实应用

---

## 1. 为什么需要 GMM？

### 🎯 OLS 的局限性

在因子模型估计中，**OLS（最小二乘法）** 要求严格的假设：

- 误差项独立同分布 (i.i.d.)
- 误差项与自变量不相关（外生性）
- 同方差

但在资产定价中，这些假设常常不成立：

| 问题 | OLS 假设 | 实际情况 |
|------|---------|----------|
| 异方差 | $\text{Var}(\epsilon_i) = \sigma^2$ | 不同资产波动率差异巨大 |
| 截面相关 | $\text{Cov}(\epsilon_i, \epsilon_j) = 0$ | 资产收益率普遍相关 |
| 生成的回归量 | $X$ 为固定或外生 | $\beta$ 是第一步回归的估计值 |
| 模型检验 | 无内置检验 | 需要检验定价误差是否联合为零 |

### 💡 GMM 的优势

**广义矩估计 (Generalized Method of Moments, GMM)** 是一个非常灵活的框架：

1. **不需要知道完整的分布** —— 只需要指定矩条件
2. **允许异方差和自相关** —— 通过合适的权重矩阵处理
3. **内置模型检验** —— J 检验直接检验模型是否被数据支持
4. **统一框架** —— OLS、IV、MLE 都是 GMM 的特殊情况

### 📖 书中的定义

> 广义矩估计是因子模型检验的标准框架。它的核心思想是：
> 利用经济理论提供的矩条件（总体矩为零的条件），通过最小化样本矩来估计参数。
> 当矩条件个数多于待估参数个数时，GMM 还提供了检验模型设定的工具（J 检验）。

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 样本均值的方差：GMM 的数学基础

### 📐 从 5 个数说起

假设我们观测到 5 个 $u_t$ 值：0, -1, 3, 3, -3

样本均值 $\bar{u} = \frac{1}{5}(0 + (-1) + 3 + 3 + (-3)) = 0.4$

但 $\bar{u}$ 本身是一个**随机变量**——不同的样本会产生不同的 $\bar{u}$！

### 📐 样本均值的方差推导

$$\text{Var}(\bar{u}) = \text{Var}\left(\frac{1}{T}\sum_{t=1}^{T} u_t\right) = \frac{1}{T^2} \sum_{i=1}^{T} \sum_{j=1}^{T} \text{Cov}(u_i, u_j)$$

**情况 1**：当 $u_t$ 独立同分布 (i.i.d.) 时

$$\text{Var}(\bar{u}) = \frac{\sigma^2}{T}$$

这是我们熟悉的公式。

**情况 2**：当 $u_t$ 存在自相关时

$$\text{Var}(\bar{u}) = \frac{S}{T}$$

其中 $S = \sum_{j=-\infty}^{\infty} E[u_t \cdot u_{t-j}]$ 是 **谱密度矩阵 (spectral density matrix)**

### 🎯 手算：有自相关的序列的样本均值方差

下面我们用 AR(1) 过程 $u_t = \rho \cdot u_{t-1} + \varepsilon_t$ 来演示：
- 理论上 $S = \sigma_\varepsilon^2 / (1 - \rho^2)$（一阶自回归的谱密度）
- 朴素公式 $\sigma^2 / T$ 会**低估**真实方差

In [ ]:
# ========== 样本均值的方差：AR(1) 过程演示 ==========
np.random.seed(42)

# --- 参数设置 ---
T = 200           # 时间序列长度
rho = 0.8         # AR(1) 系数（自相关系数）
sigma_eps = 1.0   # 创新项标准差

# --- 生成 AR(1) 过程: u_t = rho * u_{t-1} + eps_t ---
eps = np.random.normal(0, sigma_eps, T)
u = np.zeros(T)
u[0] = eps[0]
for t in range(1, T):
    u[t] = rho * u[t-1] + eps[t]

# --- 样本均值 ---
u_bar = u.mean()

print("📋 AR(1) 过程参数:")
print(f"  T = {T}, ρ = {rho}, σ_ε = {sigma_eps}")
print(f"  样本均值 ū = {u_bar:.4f}")
print(f"  样本方差 Var(u) = {u.var():.4f}")
print(f"  理论方差 σ² = σ²_ε/(1-ρ²) = {sigma_eps**2/(1-rho**2):.4f}")

# --- 理论上的样本均值方差 ---
# 朴素公式（假设 iid）: σ²/T
var_naive = sigma_eps**2 / (1 - rho**2) / T

# 谱密度修正公式: S/T
# 对于 AR(1): S = σ²_ε / (1-ρ²)  (长期方差)
# 更精确: S = σ²_ε / (1-ρ)²  （考虑自相关的累积效应）
# 实际上 S = Σ_k γ(k) = σ²_ε/(1-ρ²) * (1+2ρ/(1-ρ)) 但简化理解为 S/T
S_theory = sigma_eps**2 / (1 - rho**2)  # 长期方差
var_corrected = S_theory / T

print(f"\n📊 样本均值 ū 的理论方差:")
print(f"  朴素公式 (iid 假设): σ²/T = {var_naive:.6f}")
print(f"  修正公式 (自相关):   S/T   = {var_corrected:.6f}")
print(f"  比值: 修正/朴素 = {var_corrected/var_naive:.2f}x")
print(f"\n  💡 忽略自相关会严重低估样本均值的方差！")

In [ ]:
# ========== 重采样 1000 次：验证理论公式 ==========
np.random.seed(42)
n_sim = 1000
T_sim = 200

# 重采样 1000 次 AR(1) 过程
sample_means = np.zeros(n_sim)

for i in range(n_sim):
    eps_sim = np.random.normal(0, sigma_eps, T_sim)
    u_sim = np.zeros(T_sim)
    u_sim[0] = eps_sim[0]
    for t in range(1, T_sim):
        u_sim[t] = rho * u_sim[t-1] + eps_sim[t]
    sample_means[i] = u_sim.mean()

# 模拟方差 vs 理论方差
var_simulated = sample_means.var()

print("📊 重采样验证 (1000 次模拟):")
print(f"  模拟得到的 Var(ū)   = {var_simulated:.6f}")
print(f"  朴素公式 σ²/T       = {var_naive:.6f}")
print(f"  修正公式 S/T        = {var_corrected:.6f}")
print(f"\n  模拟 vs 朴素: {var_simulated/var_naive:.2f}x 差异")
print(f"  模拟 vs 修正: {var_simulated/var_corrected:.2f}x 差异 (应该接近 1)")

In [ ]:
# ========== 可视化：重采样分布 vs 理论分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 样本均值的分布 ---
ax1 = axes[0]
ax1.hist(sample_means, bins=40, density=True, alpha=0.7, color='steelblue',
         edgecolor='black', label='模拟分布 (1000 次)')

# iid 假设下的理论分布
x_range = np.linspace(sample_means.min() - 0.1, sample_means.max() + 0.1, 200)
y_naive = stats.norm.pdf(x_range, 0, np.sqrt(var_naive))
y_corrected = stats.norm.pdf(x_range, 0, np.sqrt(var_corrected))

ax1.plot(x_range, y_naive, 'r--', linewidth=2, label=f'朴素公式 (σ²/T = {var_naive:.4f})')
ax1.plot(x_range, y_corrected, 'g-', linewidth=2, label=f'修正公式 (S/T = {var_corrected:.4f})')

ax1.set_xlabel('$\\bar{u}$', fontsize=12)
ax1.set_ylabel('概率密度', fontsize=12)
ax1.set_title('样本均值 $\\bar{u}$ 的分布：朴素 vs 修正', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 右图: AR(1) 过程的自相关函数 ---
ax2 = axes[1]
max_lag = 20
acf_theory = [rho**k for k in range(max_lag + 1)]
ax2.bar(range(max_lag + 1), acf_theory, color='steelblue', alpha=0.7, edgecolor='black')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('滞后阶数 k', fontsize=12)
ax2.set_ylabel('自相关系数 $\\rho^k$', fontsize=12)
ax2.set_title(f'AR(1) 自相关函数 (ρ = {rho})', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 关键直觉:")
print(f"  1. 红色虚线（朴素公式）太窄 —— 低估了样本均值的真实不确定性")
print(f"  2. 绿色实线（修正公式）与模拟分布吻合 —— 自相关确实放大了方差")
print(f"  3. 'var(ū) → S/T' 是所有 GMM 数学的基石！")
print(f"     不要小看这个简单公式！")

---

## 3. 矩条件的直觉

### 🎯 什么是"矩"？

**日常类比**：你有一袋苹果，想知道平均重量。你称了 10 个苹果，加起来除以 10。
这个"平均"就是一个**矩 (moment)**。

在统计学中，矩就是随机变量的某种期望：

| 矩 | 定义 | 含义 |
|---|------|------|
| 一阶矩 | $E[X] = \mu$ | 均值 |
| 二阶中心矩 | $E[(X - \mu)^2] = \sigma^2$ | 方差 |
| 高阶矩 | $E[(X - \mu)^k]$ | 偏度、峰度等 |

### 📐 矩条件 (Moment Conditions)

矩条件就是把矩**改写成"等于零"的形式**：

$$E[g(X, \theta)] = 0$$

其中：
- $g$ 是一个关于数据 $X$ 和参数 $\theta$ 的函数
- 当 $\theta$ 取真实值时，$g$ 的期望为零

**例子**：估计均值 $\mu$

$$E[X - \mu] = 0 \quad \Rightarrow \quad g(X, \mu) = X - \mu$$

**例子**：同时估计均值 $\mu$ 和方差 $\sigma^2$

$$E\begin{bmatrix} X - \mu \\ (X - \mu)^2 - \sigma^2 \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$$

### 💡 关键直觉

矩条件的思路是：
1. **理论**告诉我们某个表达式的期望应该为零
2. 用**样本均值**替代期望：$\bar{g}_T(\theta) = \frac{1}{T}\sum_{t=1}^{T} g(X_t, \theta)$
3. 找到让 $\bar{g}_T(\theta)$ **尽可能接近零**的 $\theta$ 值

In [ ]:
# ========== 微型例子：用矩条件估计均值和方差 ==========
# 10 个数据点，手算每一步
np.random.seed(42)
data = np.round(np.random.normal(loc=5.0, scale=2.0, size=10), 2)

print("📋 微型数据集（10 个数据点）：")
print(f"  X = {list(data)}")
print(f"  真实参数: μ = 5.0, σ² = 4.0")

# ========== 矩条件 1: E[X - μ] = 0 ==========
print(f"\n📊 步骤 1: 矩条件 1 —— E[X - μ] = 0")
print(f"  样本类比: (1/T) Σ(X_t - μ̂) = 0")
print(f"  解出: μ̂ = (1/T) Σ X_t = 样本均值")

mu_hat = data.mean()
print(f"\n  计算: μ̂ = ({' + '.join([str(x) for x in data])}) / {len(data)}")
print(f"          = {data.sum():.2f} / {len(data)}")
print(f"          = {mu_hat:.4f}")

# ========== 矩条件 2: E[(X - μ)² - σ²] = 0 ==========
print(f"\n📊 步骤 2: 矩条件 2 —— E[(X - μ)² - σ²] = 0")
print(f"  样本类比: (1/T) Σ(X_t - μ̂)² - σ̂² = 0")
print(f"  解出: σ̂² = (1/T) Σ(X_t - μ̂)²")

deviations = data - mu_hat
sq_devs = deviations**2
sigma2_hat = sq_devs.mean()  # GMM 估计量（总体方差，除以 T）

print(f"\n  各数据点的偏差和偏差平方:")
for i, (x, d, s) in enumerate(zip(data, deviations, sq_devs)):
    print(f"    X_{i+1} = {x:>5.2f}, X_{i+1} - μ̂ = {d:>6.3f}, (X_{i+1} - μ̂)² = {s:>7.4f}")

print(f"\n  σ̂² = {sq_devs.sum():.4f} / {len(data)} = {sigma2_hat:.4f}")

# ========== 验证矩条件是否满足 ==========
print("\n📊 步骤 3: 验证矩条件是否'几乎为零'")
g1 = (data - mu_hat).mean()  # 应该精确为 0
g2 = ((data - mu_hat)**2 - sigma2_hat).mean()  # 应该精确为 0
print(f"  ḡ₁ = (1/T) Σ(X_t - μ̂)           = {g1:.10f}")
print(f"  ḡ₂ = (1/T) Σ[(X_t - μ̂)² - σ̂²]  = {g2:.10f}")
print("\n  💡 两个矩条件都精确为零 —— 这就是'恰好识别'的情况")
print("     2 个矩条件，2 个参数 → 唯一解")

In [ ]:
# ========== 验证: 与传统公式完全一致 ==========
print("🔬 验证: 矩估计 vs 传统公式")
print(f"  矩估计 μ̂      = {mu_hat:.6f}")
print(f"  np.mean(data)  = {np.mean(data):.6f}")
print(f"  矩估计 σ̂²     = {sigma2_hat:.6f}")
print(f"  np.var(data)   = {np.var(data):.6f}  (ddof=0, 总体方差)")
print(f"  np.var(data,1) = {np.var(data, ddof=1):.6f}  (ddof=1, 样本方差)")
print(f"\n  ✅ 验证通过！矩估计的均值就是样本均值，方差就是总体方差公式（除以 T）")
print(f"\n  💡 注意: GMM 的方差估计量除以 T（而非 T-1），这是矩估计的自然结果")
print(f"     当 T 很大时，T 和 T-1 的差异可以忽略")

---

## 4. GMM 的一般框架

### 📐 从矩条件到 GMM

当矩条件个数 $m$ **大于**参数个数 $k$ 时，我们无法让所有矩条件同时精确为零。
这时需要一个**目标函数**来权衡各矩条件：

$$\hat{\theta}_{GMM} = \arg\min_{\theta} \; \bar{g}_T(\theta)' \; W \; \bar{g}_T(\theta)$$

其中：
- $\bar{g}_T(\theta) = \frac{1}{T}\sum_{t=1}^{T} g(X_t, \theta)$ 是样本矩的 $m \times 1$ 向量
- $W$ 是 $m \times m$ 的**正定权重矩阵**
- 目标函数是一个标量：$J(\theta) = \bar{g}_T(\theta)' W \bar{g}_T(\theta) \geq 0$

### 💡 直觉理解

把 $\bar{g}_T(\theta)$ 想象成 $m$ 个"违反程度"：
- 每个矩条件理论上应该为零
- 样本中不可能都精确为零
- GMM 找的是让这些"违反程度"的**加权平方和最小**的参数

### 📐 识别条件

| 情况 | 矩条件 vs 参数 | 名称 | 特点 |
|------|---------------|------|------|
| $m = k$ | 矩条件 = 参数 | 恰好识别 (Just-identified) | 唯一解，$W$ 不影响结果 |
| $m > k$ | 矩条件 > 参数 | 过度识别 (Over-identified) | $W$ 影响结果，可做 J 检验 |
| $m < k$ | 矩条件 < 参数 | 不可识别 (Under-identified) | 无法估计，需要更多矩条件 |

In [ ]:
# ========== 微型 GMM: 10 个数据点，简单线性模型 ==========
# 模型: Y = a + b*X + e
# 矩条件（OLS 的矩条件形式）:
#   E[e]     = E[Y - a - b*X] = 0           (截距条件)
#   E[X*e]   = E[X*(Y - a - b*X)] = 0       (斜率条件)
#   E[X²*e]  = E[X²*(Y - a - b*X)] = 0      (额外的矩条件)

np.random.seed(42)
N = 10
X = np.round(np.linspace(1, 10, N), 1)
a_true, b_true = 2.0, 0.8
epsilon = np.round(np.random.normal(0, 1.0, N), 2)
Y = a_true + b_true * X + epsilon

print("📋 微型数据集（10 个数据点）：")
print(f"  真实模型: Y = {a_true} + {b_true}*X + e")
print(f"  X = {list(X)}")
print(f"  Y = {list(np.round(Y, 2))}")
print(f"  e = {list(epsilon)}")

# ========== 定义矩条件函数 ==========
def moment_conditions(theta, X, Y):
    """计算 3 个矩条件的样本均值 (m=3 > k=2 → 过度识别)"""
    a, b = theta
    e = Y - a - b * X  # 残差
    g1 = e.mean()             # E[e] = 0
    g2 = (X * e).mean()       # E[X*e] = 0
    g3 = (X**2 * e).mean()    # E[X²*e] = 0
    return np.array([g1, g2, g3])

# ========== GMM 目标函数 ==========
def gmm_objective(theta, X, Y, W):
    """J(θ) = ḡ(θ)' W ḡ(θ)"""
    g_bar = moment_conditions(theta, X, Y)
    return g_bar @ W @ g_bar

print(f"\n📊 GMM 设置:")
print(f"  参数个数 k = 2 (a, b)")
print(f"  矩条件个数 m = 3")
print(f"  自由度 = m - k = 1")
print(f"  → 过度识别 (over-identified)")
print(f"\n  📐 矩条件:")
print(f"    g₁ = E[Y - a - bX] = 0")
print(f"    g₂ = E[X(Y - a - bX)] = 0")
print(f"    g₃ = E[X²(Y - a - bX)] = 0")

# 在真实参数处检查矩条件
g_true = moment_conditions([a_true, b_true], X, Y)
print(f"\n  在真实参数 (a={a_true}, b={b_true}) 处的样本矩:")
print(f"    ḡ₁ = {g_true[0]:.6f}")
print(f"    ḡ₂ = {g_true[1]:.6f}")
print(f"    ḡ₃ = {g_true[2]:.6f}")
print(f"\n  💡 即使在真实参数处，样本矩也不精确为零 —— 这就是为什么需要'最小化'")

In [ ]:
# ========== 可视化 GMM 目标函数 ==========
# 固定 W = I，画出 J(a, b) 的等高线
W_identity = np.eye(3)

a_grid = np.linspace(0, 4, 100)
b_grid = np.linspace(0, 1.6, 100)
A, B = np.meshgrid(a_grid, b_grid)
J_vals = np.zeros_like(A)

for i in range(len(b_grid)):
    for j in range(len(a_grid)):
        J_vals[i, j] = gmm_objective([A[i, j], B[i, j]], X, Y, W_identity)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图: 等高线图 ---
ax1 = axes[0]
cs = ax1.contourf(A, B, np.log(J_vals + 1e-10), levels=30, cmap='viridis')
ax1.contour(A, B, np.log(J_vals + 1e-10), levels=15, colors='white', linewidths=0.5, alpha=0.5)
ax1.plot(a_true, b_true, 'r*', markersize=15, label=f'True ({a_true}, {b_true})', zorder=5)

# 找到 GMM 估计值
result_1st = minimize(gmm_objective, x0=[0, 0], args=(X, Y, W_identity), method='Nelder-Mead')
a_gmm, b_gmm = result_1st.x
ax1.plot(a_gmm, b_gmm, 'wo', markersize=10, markeredgecolor='black', linewidth=2,
         label=f'GMM ({a_gmm:.2f}, {b_gmm:.2f})', zorder=5)

plt.colorbar(cs, ax=ax1, label='log J(a, b)')
ax1.set_xlabel('a (intercept)', fontsize=12)
ax1.set_ylabel('b (slope)', fontsize=12)
ax1.set_title('GMM Objective Function (W = I)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)

# --- 右图: 固定 a = a_gmm, 画 J vs b ---
ax2 = axes[1]
b_range = np.linspace(0, 1.6, 200)
J_profile = [gmm_objective([a_gmm, bi], X, Y, W_identity) for bi in b_range]
ax2.plot(b_range, J_profile, 'b-', linewidth=2)
ax2.axvline(x=b_gmm, color='red', linestyle='--', linewidth=2, label=f'b_GMM = {b_gmm:.4f}')
ax2.axvline(x=b_true, color='green', linestyle='--', linewidth=2, label=f'b_true = {b_true}')
ax2.set_xlabel('b (slope)', fontsize=12)
ax2.set_ylabel('J(a_GMM, b)', fontsize=12)
ax2.set_title(f'Profile of J (a fixed at {a_gmm:.2f})', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：GMM 目标函数的等高线。颜色越深 = J 越小 = 矩条件违反越少")
print(f"       红星 = 真实参数，白圈 = GMM 估计值")
print(f"  右图：固定 a 后，J 关于 b 的剖面图。最低点就是 GMM 的解")

---

## 5. 一阶段 GMM vs 两阶段 GMM

### 📐 权重矩阵 $W$ 的选择

**一阶段 GMM**：使用简单的权重矩阵，最常用的是单位矩阵

$$W_1 = I_m$$

**两阶段 GMM**：使用最优权重矩阵

$$W_2 = \hat{S}^{-1}$$

其中 $\hat{S}$ 是样本矩 $g(X_t, \hat{\theta}_1)$ 的长期协方差矩阵（long-run variance）：

$$\hat{S} = \frac{1}{T} \sum_{t=1}^{T} g_t(\hat{\theta}_1) \, g_t(\hat{\theta}_1)'$$

### 💡 为什么最优权重矩阵是 $S^{-1}$？

直觉：
- 方差大的矩条件"噪声"多，信息量少 → 应该给**较低权重**
- 方差小的矩条件"精确"，信息量大 → 应该给**较高权重**
- $S^{-1}$ 正好实现了这种"反方差加权"

数学上，Hansen (1982) 证明了：使用 $W = S^{-1}$ 时，GMM 估计量具有**最小渐近方差**。

### 📐 两阶段 GMM 步骤

```
第一阶段:
  W₁ = I
  θ̂₁ = argmin ḡ(θ)' I ḡ(θ)
  
计算最优权重:
  Ŝ = (1/T) Σ g_t(θ̂₁) g_t(θ̂₁)'
  W₂ = Ŝ⁻¹

第二阶段:
  θ̂₂ = argmin ḡ(θ)' W₂ ḡ(θ)
```

In [ ]:
# ========== 一阶段 GMM (W = I) ==========
print("📊 步骤 1: 一阶段 GMM")
print("═" * 55)

W1 = np.eye(3)
print(f"  权重矩阵 W₁ = I₃ (单位矩阵)")
print(f"  {W1}")

result_1st = minimize(gmm_objective, x0=[0, 0], args=(X, Y, W1), method='Nelder-Mead')
theta_1st = result_1st.x
a1, b1 = theta_1st

print(f"\n  一阶段 GMM 估计:")
print(f"    â₁ = {a1:.6f}  (真实值: {a_true})")
print(f"    b̂₁ = {b1:.6f}  (真实值: {b_true})")

# 检查一阶段矩条件
g_1st = moment_conditions(theta_1st, X, Y)
print(f"\n  一阶段样本矩:")
print(f"    ḡ₁ = {g_1st[0]:.8f}")
print(f"    ḡ₂ = {g_1st[1]:.8f}")
print(f"    ḡ₃ = {g_1st[2]:.8f}")
print(f"\n  💡 过度识别时，矩条件不会精确为零，只是尽量接近零")

In [ ]:
# ========== 计算最优权重矩阵 ==========
print("📊 步骤 2: 计算最优权重矩阵")
print("═" * 55)

# 计算每个数据点的矩条件值 (T × m 矩阵的每一行)
e_1st = Y - a1 - b1 * X  # 一阶段残差
G = np.column_stack([e_1st, X * e_1st, X**2 * e_1st])  # T × 3

print(f"  每个数据点的矩条件值 g_t (T={N} × m=3):")
for i in range(N):
    print(f"    t={i+1}: g = [{G[i,0]:>8.4f}, {G[i,1]:>8.4f}, {G[i,2]:>10.4f}]")

# 计算 S = (1/T) * G'G
S_hat = G.T @ G / N
print(f"\n  长期协方差矩阵 Ŝ = (1/T) Σ g_t g_t':")
print(f"  {np.array2string(S_hat, precision=4, suppress_small=True)}")

# 最优权重矩阵
W2 = np.linalg.inv(S_hat)
print(f"\n  最优权重矩阵 W₂ = Ŝ⁻¹:")
print(f"  {np.array2string(W2, precision=4, suppress_small=True)}")

print(f"\n  💡 对角线元素: [{W2[0,0]:.4f}, {W2[1,1]:.4f}, {W2[2,2]:.4f}]")
print(f"     方差大的矩条件对应较小的权重，方差小的矩条件对应较大的权重")

In [ ]:
# ========== 两阶段 GMM ==========
print("📊 步骤 3: 两阶段 GMM")
print("═" * 55)

result_2nd = minimize(gmm_objective, x0=theta_1st, args=(X, Y, W2), method='Nelder-Mead')
theta_2nd = result_2nd.x
a2, b2 = theta_2nd

print(f"  两阶段 GMM 估计:")
print(f"    â₂ = {a2:.6f}  (真实值: {a_true})")
print(f"    b̂₂ = {b2:.6f}  (真实值: {b_true})")

# 两阶段矩条件
g_2nd = moment_conditions(theta_2nd, X, Y)
print(f"\n  两阶段样本矩:")
print(f"    ḡ₁ = {g_2nd[0]:.8f}")
print(f"    ḡ₂ = {g_2nd[1]:.8f}")
print(f"    ḡ₃ = {g_2nd[2]:.8f}")

# ========== 对比 ==========
print(f"\n📊 步骤 4: 一阶段 vs 两阶段对比")
print("─" * 55)
print(f"  {'':>15} {'一阶段':>10} {'两阶段':>10} {'真实值':>10}")
print(f"  {'─'*45}")
print(f"  {'â (截距)':>15} {a1:>10.4f} {a2:>10.4f} {a_true:>10.4f}")
print(f"  {'b̂ (斜率)':>15} {b1:>10.4f} {b2:>10.4f} {b_true:>10.4f}")
print(f"  {'J (目标函数)':>15} {result_1st.fun:>10.6f} {result_2nd.fun:>10.6f} {'':>10}")

# OLS 对比
from numpy.polynomial import polynomial as P
b_ols, a_ols = np.polyfit(X, Y, 1)
print(f"\n  OLS 估计: â = {a_ols:.4f}, b̂ = {b_ols:.4f}")
print(f"\n  💡 在这个简单例子中，GMM 和 OLS 的估计值非常接近")
print(f"     但 GMM 的优势在于：(1) 可以处理更多矩条件 (2) 可以做 J 检验")

In [ ]:
# ========== 计算 GMM 标准误 ==========
print("📊 步骤 5: GMM 标准误估计")
print("═" * 55)

# GMM 标准误: Var(θ̂) = (1/T) * (D'WD)⁻¹ D'W S W D (D'WD)⁻¹
# 其中 D = ∂ḡ/∂θ' (m × k 的 Jacobian 矩阵)
# 在最优 W = S⁻¹ 时简化为: Var(θ̂) = (1/T) * (D'S⁻¹D)⁻¹

# 数值计算 Jacobian D
delta = 1e-6
D = np.zeros((3, 2))  # m × k
for j in range(2):
    theta_plus = theta_2nd.copy()
    theta_minus = theta_2nd.copy()
    theta_plus[j] += delta
    theta_minus[j] -= delta
    D[:, j] = (moment_conditions(theta_plus, X, Y) - moment_conditions(theta_minus, X, Y)) / (2 * delta)

print(f"  Jacobian 矩阵 D = ∂ḡ/∂θ' (3 × 2):")
print(f"  {np.array2string(D, precision=6)}")

# 更新 S 用两阶段残差
e_2nd = Y - a2 - b2 * X
G2 = np.column_stack([e_2nd, X * e_2nd, X**2 * e_2nd])
S2 = G2.T @ G2 / N

# 两阶段标准误 (最优权重)
V_opt = np.linalg.inv(D.T @ np.linalg.inv(S2) @ D) / N
se_2nd = np.sqrt(np.diag(V_opt))

# 一阶段标准误 (sandwich formula)
V_1st = np.linalg.inv(D.T @ W1 @ D) @ (D.T @ W1 @ S2 @ W1 @ D) @ np.linalg.inv(D.T @ W1 @ D) / N
se_1st = np.sqrt(np.diag(V_1st))

print(f"\n  标准误对比:")
print(f"  {'':>15} {'一阶段 SE':>12} {'两阶段 SE':>12} {'效率增益':>12}")
print(f"  {'─'*52}")
print(f"  {'â (截距)':>15} {se_1st[0]:>12.6f} {se_2nd[0]:>12.6f} {(1 - se_2nd[0]/se_1st[0])*100:>10.1f}%")
print(f"  {'b̂ (斜率)':>15} {se_1st[1]:>12.6f} {se_2nd[1]:>12.6f} {(1 - se_2nd[1]/se_1st[1])*100:>10.1f}%")

print(f"\n  💡 两阶段 GMM 使用最优权重矩阵，标准误通常更小（效率更高）")
print(f"     效率增益 = 1 - SE₂/SE₁，正值表示两阶段更精确")

---

## 6. J 检验（过度识别检验）

### 🎯 核心问题

当矩条件个数 $m$ 大于参数个数 $k$ 时，我们有"多余"的矩条件。
如果模型是正确的，这些多余的矩条件在大样本下也应该接近零。

**J 检验（Hansen's test）** 就是检验这些多余矩条件是否"足够接近零"。

### 📐 J 统计量

$$J = T \cdot \bar{g}_T(\hat{\theta})' \hat{W} \bar{g}_T(\hat{\theta}) \xrightarrow{d} \chi^2(m - k)$$

其中：
- $T$ = 样本量
- $\hat{\theta}$ = 两阶段 GMM 估计值
- $\hat{W} = \hat{S}^{-1}$ = 最优权重矩阵
- $m - k$ = 过度识别的自由度

### 💡 解读

| J 检验结果 | 含义 |
|-----------|------|
| P 值 > 0.05 | 不拒绝模型 → 矩条件与数据一致，模型设定可接受 |
| P 值 < 0.05 | 拒绝模型 → 矩条件与数据矛盾，模型可能误设 |

### ⚠️ 注意

- J 检验只能在**过度识别** ($m > k$) 时使用
- 恰好识别 ($m = k$) 时，矩条件可以精确满足，J = 0，无法检验
- J 检验是"不拒绝"而非"证明正确" —— 多个不同模型可能都通过 J 检验

In [ ]:
# ========== J 检验: 手动计算 ==========
print("📊 步骤 1: 手动计算 J 统计量")
print("═" * 55)

# 两阶段 GMM 的样本矩
g_bar = moment_conditions(theta_2nd, X, Y)
print(f"  两阶段 GMM 的样本矩 ḡ(θ̂₂):")
print(f"    ḡ₁ = {g_bar[0]:.8f}")
print(f"    ḡ₂ = {g_bar[1]:.8f}")
print(f"    ḡ₃ = {g_bar[2]:.8f}")

# J = T * ḡ' Ŝ⁻¹ ḡ
J_stat = N * g_bar @ W2 @ g_bar
print(f"\n  📐 J = T × ḡ' Ŝ⁻¹ ḡ")
print(f"       = {N} × {g_bar @ W2 @ g_bar:.8f}")
print(f"       = {J_stat:.6f}")

# 自由度
m = 3  # 矩条件个数
k = 2  # 参数个数
df_J = m - k
print(f"\n  自由度 = m - k = {m} - {k} = {df_J}")

# P 值
p_J = 1 - stats.chi2.cdf(J_stat, df=df_J)
print(f"\n📊 步骤 2: 计算 P 值")
print(f"  J ~ χ²({df_J})")
print(f"  P(χ² > {J_stat:.4f}) = {p_J:.6f}")

# 结论
print(f"\n📊 步骤 3: 结论")
alpha = 0.05
if p_J > alpha:
    print(f"  ✅ P = {p_J:.4f} > {alpha}")
    print(f"  ✅ 不拒绝 H₀: 模型设定可接受，矩条件与数据一致")
else:
    print(f"  ❌ P = {p_J:.4f} < {alpha}")
    print(f"  ❌ 拒绝 H₀: 模型设定可能有问题")

In [ ]:
# ========== 用 scipy 验证 J 检验 ==========
chi2_critical = stats.chi2.ppf(0.95, df=df_J)
p_verify = 1 - stats.chi2.cdf(J_stat, df=df_J)

print("🔬 scipy.stats.chi2 验证:")
print(f"  手算 J 统计量  = {J_stat:.6f}")
print(f"  χ²({df_J}) 临界值 (95%) = {chi2_critical:.6f}")
print(f"  手算 P 值      = {p_J:.6f}")
print(f"  scipy P 值     = {p_verify:.6f}")
print(f"\n  ✅ 验证通过！")

In [ ]:
# ========== 可视化 J 检验 ==========
fig, ax = plt.subplots(figsize=(10, 6))

x_chi2 = np.linspace(0, 12, 500)
y_chi2 = stats.chi2.pdf(x_chi2, df=df_J)

# 分布曲线
ax.plot(x_chi2, y_chi2, 'b-', linewidth=2, label=f'$\\chi^2$({df_J}) Distribution')

# 拒绝域
ax.fill_between(x_chi2, y_chi2, where=(x_chi2 >= chi2_critical),
                color='#e74c3c', alpha=0.4, label=f'Rejection Region (> {chi2_critical:.2f})')

# 接受域
ax.fill_between(x_chi2, y_chi2, where=(x_chi2 < chi2_critical),
                color='#2ecc71', alpha=0.2, label='Non-rejection Region')

# J 统计量位置
ax.axvline(x=J_stat, color='red', linestyle='--', linewidth=2,
           label=f'J = {J_stat:.4f} (P = {p_J:.4f})')

# 临界值
ax.axvline(x=chi2_critical, color='orange', linestyle=':', linewidth=2,
           label=f'Critical Value = {chi2_critical:.2f}')

ax.set_xlabel('J Statistic', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title(f'J-Test: Overidentifying Restrictions (df = {df_J})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# 注释框
result_text = 'PASS' if p_J > 0.05 else 'FAIL'
textstr = f'J = {J_stat:.4f}\nP-value = {p_J:.4f}\nResult: {result_text}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.98, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  蓝色曲线: χ²({df_J}) 分布")
print(f"  红色虚线: 我们的 J 统计量 = {J_stat:.4f}")
print(f"  橙色虚线: 95% 临界值 = {chi2_critical:.2f}")
print(f"  J 落在绿色区域（非拒绝域）→ 模型通过 J 检验")

---

## 7. GMM 在因子模型中的应用

### 🎯 资产定价的矩条件

在因子模型中，核心定价方程为：

$$E[R_i^e] = \beta_i' \lambda$$

其中：
- $R_i^e = R_i - R_f$ 是资产 $i$ 的超额收益
- $\beta_i$ 是资产 $i$ 对因子的暴露（$K \times 1$ 向量）
- $\lambda$ 是因子风险溢价（$K \times 1$ 向量，待估参数）

### 📐 GMM 矩条件的构建

对于 $N$ 个资产和 $K$ 个因子的模型，定义**定价误差 (pricing error)**：

$$\alpha_i = E[R_i^e] - \beta_i' \lambda$$

如果模型正确，所有定价误差应为零：$\alpha_i = 0, \; \forall i$。

GMM 的矩条件为：

$$E[R_{i,t}^e - \beta_i' \lambda] = 0, \quad i = 1, 2, \ldots, N$$

这里有 $N$ 个矩条件（每个资产一个），$K$ 个参数（因子风险溢价）。

当 $N > K$ 时，模型是过度识别的，可以做 J 检验。

### 📖 书中要点

> GMM 在因子模型检验中的核心价值在于：
> 它同时完成了两个任务——估计因子风险溢价 $\lambda$，以及检验定价误差 $\alpha$ 是否联合为零。
> J 统计量在 $H_0: \alpha = 0$ 下服从 $\chi^2(N - K)$ 分布。

### 📐 完整流程

```
第一步: 时间序列回归
  对每个资产 i: R_i,t - Rf = α_i + β_i * f_t + ε_i,t
  → 得到 β̂_i (N × K 矩阵)

第二步: GMM 估计 λ
  矩条件: E[R_i^e - β_i'λ] = 0
  等价于: 最小化 α'Wα, 其中 α = R̄^e - β̂λ
  
第三步: J 检验
  J = T * α̂' Ŝ⁻¹ α̂ ~ χ²(N - K)
  → 检验定价误差是否联合显著为零
```

In [ ]:
# ========== 生成因子模型数据 ==========
np.random.seed(42)

# --- 模型参数 ---
N_ASSETS = 25       # 资产数量 (25 个组合，类似 FF 25)
T_MONTHS = 120      # 时间序列长度 (10 年月度数据)
K_FACTORS = 1       # 因子个数 (单因子模型)

# 真实参数
lambda_true = 0.5   # 因子风险溢价 (% per month)
sigma_f = 4.0       # 因子收益率标准差

# 各资产的 beta (均匀分布在 0.5 到 1.5)
betas_true = np.linspace(0.5, 1.5, N_ASSETS)

print(f"📊 模拟参数：")
print(f"  资产数量 N = {N_ASSETS}")
print(f"  月份数量 T = {T_MONTHS}")
print(f"  因子数量 K = {K_FACTORS}")
print(f"  真实因子风险溢价 λ = {lambda_true}% / 月")
print(f"  因子收益率标准差 σ_f = {sigma_f}%")
print(f"  β 范围: [{betas_true.min():.2f}, {betas_true.max():.2f}]")

# --- 生成因子收益率 ---
# f_t ~ N(λ, σ_f²)
factor_returns = np.random.normal(lambda_true, sigma_f, T_MONTHS)

# --- 生成资产超额收益 ---
# R_i,t = β_i * f_t + ε_i,t
# 注意：E[R_i] = β_i * E[f] = β_i * λ (定价方程自动满足)
sigma_eps = 3.0  # 特异性波动率
excess_returns = np.zeros((T_MONTHS, N_ASSETS))

for i in range(N_ASSETS):
    excess_returns[:, i] = betas_true[i] * factor_returns + np.random.normal(0, sigma_eps, T_MONTHS)

print(f"\n📊 数据概况:")
print(f"  因子收益率: 均值 = {factor_returns.mean():.4f}%, 标准差 = {factor_returns.std():.4f}%")
print(f"  各资产平均超额收益范围: [{excess_returns.mean(axis=0).min():.4f}%, {excess_returns.mean(axis=0).max():.4f}%]")
print(f"\n  DGP (数据生成过程):")
print(f"    f_t ~ N(λ={lambda_true}, σ²_f={sigma_f}²)")
print(f"    R_i,t = β_i × f_t + ε_i,t, ε_i,t ~ N(0, {sigma_eps}²)")
print(f"    → E[R_i] = β_i × λ = β_i × {lambda_true}")

In [ ]:
# ========== 第一步: 时间序列回归估计 β ==========
print("📊 步骤 1: 时间序列回归估计 β")
print("═" * 55)

betas_hat = np.zeros(N_ASSETS)
alphas_ts = np.zeros(N_ASSETS)  # 时间序列 alpha
residuals = np.zeros((T_MONTHS, N_ASSETS))

for i in range(N_ASSETS):
    # OLS: R_i,t = α_i + β_i * f_t + ε_i,t
    X_reg = np.column_stack([np.ones(T_MONTHS), factor_returns])
    coeffs = np.linalg.lstsq(X_reg, excess_returns[:, i], rcond=None)[0]
    alphas_ts[i] = coeffs[0]
    betas_hat[i] = coeffs[1]
    residuals[:, i] = excess_returns[:, i] - X_reg @ coeffs

# 展示前 5 个和后 5 个资产
print(f"  {'资产':>6} {'β_true':>8} {'β_hat':>8} {'α_ts':>8} {'E[R]':>8}")
print(f"  {'─'*40}")
mean_rets = excess_returns.mean(axis=0)
for i in list(range(5)) + list(range(N_ASSETS-5, N_ASSETS)):
    if i == 5:
        print(f"  {'...':>6}")
    print(f"  {i+1:>6d} {betas_true[i]:>8.4f} {betas_hat[i]:>8.4f} {alphas_ts[i]:>8.4f} {mean_rets[i]:>8.4f}")

# β 估计精度
beta_rmse = np.sqrt(((betas_hat - betas_true)**2).mean())
print(f"\n  β 估计的 RMSE = {beta_rmse:.6f}")
print(f"  💡 β 估计值与真实值非常接近，时间序列回归是可靠的")

In [ ]:
# ========== 第二步: GMM 估计因子风险溢价 λ ==========
print("📊 步骤 2: GMM 估计因子风险溢价 λ")
print("═" * 55)

# --- 矩条件 ---
# g_i(λ) = E[R_i^e - β̂_i * λ] = 0, i = 1, ..., N
# 样本类比: ḡ_i(λ) = R̄_i^e - β̂_i * λ = α̂_i (定价误差)

R_bar = excess_returns.mean(axis=0)  # N × 1, 各资产平均超额收益

def pricing_errors(lam, R_bar, betas):
    """计算定价误差 α = R̄ - β * λ"""
    return R_bar - betas * lam

def factor_gmm_objective(lam, R_bar, betas, W):
    """GMM 目标函数: α'Wα"""
    alpha = pricing_errors(lam, R_bar, betas)
    return alpha @ W @ alpha

# --- 一阶段 GMM: W = I ---
print("\n  --- 一阶段 GMM (W = I) ---")
W_I = np.eye(N_ASSETS)
result_stage1 = minimize(factor_gmm_objective, x0=[0], args=(R_bar, betas_hat, W_I), method='Nelder-Mead')
lambda_1st = result_stage1.x[0]
alpha_1st = pricing_errors(lambda_1st, R_bar, betas_hat)

print(f"  λ̂₁ = {lambda_1st:.6f}% (真实值: {lambda_true})")
print(f"  定价误差范围: [{alpha_1st.min():.4f}%, {alpha_1st.max():.4f}%]")
print(f"  定价误差 RMSE: {np.sqrt((alpha_1st**2).mean()):.4f}%")

# --- 解析解 (W=I 时 λ 的解析解) ---
# ḡ(λ) = R̄ - β*λ, 目标 = (R̄-βλ)'(R̄-βλ)
# FOC: -2β'(R̄ - βλ) = 0 → λ = (β'β)⁻¹ β'R̄ = OLS of R̄ on β
lambda_analytic = (betas_hat @ R_bar) / (betas_hat @ betas_hat)
print(f"\n  📐 解析解: λ̂ = (β̂'β̂)⁻¹ β̂'R̄ = {lambda_analytic:.6f}")
print(f"     与数值优化结果一致 ✅")

In [ ]:
# ========== 两阶段 GMM ==========
print("\n  --- 两阶段 GMM ---")
print("═" * 55)

# 计算最优权重矩阵
# S = (1/T) Σ (R_t^e - β*λ̂₁)(R_t^e - β*λ̂₁)'
# 其中 R_t^e - β*λ̂₁ 是 t 时刻各资产的定价误差
pricing_err_ts = excess_returns - np.outer(np.ones(T_MONTHS), betas_hat * lambda_1st)  # T × N
S_factor = pricing_err_ts.T @ pricing_err_ts / T_MONTHS  # N × N

print(f"  📊 步骤 2a: 计算长期协方差矩阵 Ŝ ({N_ASSETS} × {N_ASSETS})")
print(f"    Ŝ 的对角线元素范围: [{np.diag(S_factor).min():.4f}, {np.diag(S_factor).max():.4f}]")

# 最优权重
W_opt = np.linalg.inv(S_factor)
print(f"  📊 步骤 2b: W₂ = Ŝ⁻¹")

# 两阶段估计
result_stage2 = minimize(factor_gmm_objective, x0=[lambda_1st], args=(R_bar, betas_hat, W_opt), method='Nelder-Mead')
lambda_2nd = result_stage2.x[0]
alpha_2nd = pricing_errors(lambda_2nd, R_bar, betas_hat)

# 解析解: λ̂₂ = (β'Ŝ⁻¹β)⁻¹ β'Ŝ⁻¹R̄ = GLS
lambda_2nd_analytic = (betas_hat @ W_opt @ R_bar) / (betas_hat @ W_opt @ betas_hat)

print(f"\n  📊 步骤 2c: 两阶段 GMM 结果")
print(f"    λ̂₂ (数值) = {lambda_2nd:.6f}%")
print(f"    λ̂₂ (解析) = {lambda_2nd_analytic:.6f}%")
print(f"    定价误差 RMSE: {np.sqrt((alpha_2nd**2).mean()):.4f}%")

# --- 标准误 ---
# SE(λ̂) = sqrt[(1/T) * (β'Ŝ⁻¹β)⁻¹]
var_lambda_1st = (1/T_MONTHS) * (betas_hat @ betas_hat)**(-1) * (betas_hat @ S_factor @ betas_hat) * (betas_hat @ betas_hat)**(-1)
se_lambda_1st = np.sqrt(var_lambda_1st)
var_lambda_2nd = (1/T_MONTHS) / (betas_hat @ W_opt @ betas_hat)
se_lambda_2nd = np.sqrt(var_lambda_2nd)

print(f"\n  📊 标准误对比:")
print(f"    {'':>20} {'一阶段':>10} {'两阶段':>10} {'真实值':>10}")
print(f"    {'─'*45}")
print(f"    {'λ̂':>20} {lambda_1st:>10.4f} {lambda_2nd:>10.4f} {lambda_true:>10.4f}")
print(f"    {'SE(λ̂)':>20} {se_lambda_1st:>10.4f} {se_lambda_2nd:>10.4f} {'':>10}")
print(f"    {'t = λ̂/SE':>20} {lambda_1st/se_lambda_1st:>10.4f} {lambda_2nd/se_lambda_2nd:>10.4f} {'':>10}")

print(f"\n  💡 两阶段 GMM 使用 GLS 权重，标准误更小，估计更高效")

In [ ]:
# ========== J 检验: 因子模型检验 ==========
print("📊 步骤 3: J 检验（因子模型定价误差检验）")
print("═" * 55)

# J = T * α̂' Ŝ⁻¹ α̂ ~ χ²(N - K)
# 使用两阶段残差更新 S
pricing_err_ts_2 = excess_returns - np.outer(np.ones(T_MONTHS), betas_hat * lambda_2nd)
S2_factor = pricing_err_ts_2.T @ pricing_err_ts_2 / T_MONTHS
W2_factor = np.linalg.inv(S2_factor)

J_factor = T_MONTHS * alpha_2nd @ W2_factor @ alpha_2nd
df_factor = N_ASSETS - K_FACTORS
p_factor = 1 - stats.chi2.cdf(J_factor, df=df_factor)

print(f"  矩条件个数 m = {N_ASSETS} (每个资产一个定价误差)")
print(f"  参数个数 k = {K_FACTORS} (因子风险溢价)")
print(f"  自由度 = m - k = {N_ASSETS} - {K_FACTORS} = {df_factor}")
print(f"")
print(f"  📐 J = T × α̂' Ŝ⁻¹ α̂")
print(f"       = {T_MONTHS} × {alpha_2nd @ W2_factor @ alpha_2nd:.6f}")
print(f"       = {J_factor:.4f}")
print(f"")
print(f"  P(χ²({df_factor}) > {J_factor:.4f}) = {p_factor:.6f}")

chi2_crit = stats.chi2.ppf(0.95, df=df_factor)
print(f"  χ²({df_factor}) 的 95% 临界值 = {chi2_crit:.4f}")
print(f"")

if p_factor > 0.05:
    print(f"  ✅ J = {J_factor:.2f} < {chi2_crit:.2f}, P = {p_factor:.4f} > 0.05")
    print(f"  ✅ 不拒绝 H₀: 定价误差联合不显著")
    print(f"  ✅ 结论: 单因子模型能够较好地解释这 {N_ASSETS} 个资产的截面收益差异")
else:
    print(f"  ❌ J = {J_factor:.2f} > {chi2_crit:.2f}, P = {p_factor:.4f} < 0.05")
    print(f"  ❌ 拒绝 H₀: 定价误差联合显著不为零")
    print(f"  ❌ 结论: 单因子模型不能充分解释截面收益差异，可能需要更多因子")

In [ ]:
# ========== 可视化: 因子模型 GMM 结果 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: β vs E[R] 截面关系 ---
ax1 = axes[0]
ax1.scatter(betas_hat, R_bar * 100, c='steelblue', s=60, edgecolors='black',
            alpha=0.8, label='Assets', zorder=5)

# GMM 拟合线: E[R] = β * λ̂
beta_range = np.linspace(betas_hat.min() - 0.1, betas_hat.max() + 0.1, 100)
ax1.plot(beta_range, beta_range * lambda_2nd * 100, 'r-', linewidth=2,
         label=f'GMM: E[R] = {lambda_2nd:.3f}$\\beta$ (bps)')

# 真实关系
ax1.plot(beta_range, beta_range * lambda_true * 100, 'g--', linewidth=2,
         label=f'True: E[R] = {lambda_true}$\\beta$ (bps)')

ax1.set_xlabel('$\\hat{\\beta}$', fontsize=12)
ax1.set_ylabel('Average Excess Return (bps)', fontsize=12)
ax1.set_title('Cross-Sectional Fit: $\\beta$ vs E[R]', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 图2: 定价误差 ---
ax2 = axes[1]
colors_alpha = ['#2ecc71' if a > 0 else '#e74c3c' for a in alpha_2nd]
bars = ax2.bar(range(1, N_ASSETS+1), alpha_2nd, color=colors_alpha,
               edgecolor='black', alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Asset Index', fontsize=12)
ax2.set_ylabel('Pricing Error ($\\hat{\\alpha}$, %)', fontsize=12)
ax2.set_title(f'Pricing Errors (RMSE = {np.sqrt((alpha_2nd**2).mean()):.4f}%)',
              fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# --- 图3: J 检验分布 ---
ax3 = axes[2]
x_chi2 = np.linspace(0, max(J_factor * 2, chi2_crit * 1.5), 500)
y_chi2 = stats.chi2.pdf(x_chi2, df=df_factor)
ax3.plot(x_chi2, y_chi2, 'b-', linewidth=2, label=f'$\\chi^2$({df_factor})')
ax3.fill_between(x_chi2, y_chi2, where=(x_chi2 >= chi2_crit),
                 color='#e74c3c', alpha=0.3, label=f'Rejection (> {chi2_crit:.1f})')
ax3.axvline(x=J_factor, color='red', linestyle='--', linewidth=2,
            label=f'J = {J_factor:.2f}')

textstr = f'J = {J_factor:.2f}\ndf = {df_factor}\nP = {p_factor:.4f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax3.text(0.98, 0.98, textstr, transform=ax3.transAxes, fontsize=11,
         verticalalignment='top', horizontalalignment='right', bbox=props)

ax3.set_xlabel('J Statistic', fontsize=12)
ax3.set_ylabel('Probability Density', fontsize=12)
ax3.set_title('J-Test for Factor Model', fontsize=14, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：截面关系图 —— 如果模型正确，所有点应落在拟合线上")
print(f"       散点偏离拟合线的程度就是定价误差 α")
print(f"  图2：各资产的定价误差 —— 正负交替、幅度小，说明模型拟合良好")
print(f"  图3：J 统计量在 χ² 分布上的位置 —— 落在非拒绝域内")

In [ ]:
# ========== 对比: GMM vs Fama-MacBeth ==========
print("📊 步骤 4: GMM 与 Fama-MacBeth 回归对比")
print("═" * 55)

# --- Fama-MacBeth 回归 ---
# 每个月 t: 截面回归 R_i,t = γ_0,t + γ_1,t * β̂_i + e_i,t
# λ̂_FM = (1/T) Σ γ̂_1,t

gamma_0_ts = np.zeros(T_MONTHS)
gamma_1_ts = np.zeros(T_MONTHS)

for t in range(T_MONTHS):
    # 截面回归 (有截距)
    X_cs = np.column_stack([np.ones(N_ASSETS), betas_hat])
    coeffs_cs = np.linalg.lstsq(X_cs, excess_returns[t, :], rcond=None)[0]
    gamma_0_ts[t] = coeffs_cs[0]
    gamma_1_ts[t] = coeffs_cs[1]

lambda_fm = gamma_1_ts.mean()
se_fm = gamma_1_ts.std(ddof=1) / np.sqrt(T_MONTHS)
t_fm = lambda_fm / se_fm

# --- 无截距 Fama-MacBeth ---
gamma_1_noint = np.zeros(T_MONTHS)
for t in range(T_MONTHS):
    # 截面回归 (无截距): R_i,t = γ_1,t * β̂_i + e_i,t
    gamma_1_noint[t] = (betas_hat @ excess_returns[t, :]) / (betas_hat @ betas_hat)

lambda_fm_noint = gamma_1_noint.mean()
se_fm_noint = gamma_1_noint.std(ddof=1) / np.sqrt(T_MONTHS)

# --- 汇总对比 ---
print(f"  {'方法':>25} {'λ̂':>10} {'SE':>10} {'t 值':>10}")
print(f"  {'─'*52}")
print(f"  {'GMM 一阶段 (W=I)':>25} {lambda_1st:>10.4f} {se_lambda_1st:>10.4f} {lambda_1st/se_lambda_1st:>10.4f}")
print(f"  {'GMM 两阶段 (W=Ŝ⁻¹)':>25} {lambda_2nd:>10.4f} {se_lambda_2nd:>10.4f} {lambda_2nd/se_lambda_2nd:>10.4f}")
print(f"  {'FM 回归 (有截距)':>25} {lambda_fm:>10.4f} {se_fm:>10.4f} {t_fm:>10.4f}")
print(f"  {'FM 回归 (无截距)':>25} {lambda_fm_noint:>10.4f} {se_fm_noint:>10.4f} {lambda_fm_noint/se_fm_noint:>10.4f}")
print(f"  {'真实值':>25} {lambda_true:>10.4f} {'':>10} {'':>10}")

print(f"\n  💡 关键观察:")
print(f"     1. GMM 一阶段(W=I) ≈ FM 无截距回归 —— 它们是等价的")
print(f"     2. GMM 两阶段(W=Ŝ⁻¹) 使用了 GLS 权重，效率更高")
print(f"     3. FM 有截距回归允许 γ₀ ≠ 0，这可以捕捉模型无法解释的基础收益")
print(f"     4. GMM 的优势: 提供了 J 检验来正式检验模型设定")

In [ ]:
# ========== 可视化: GMM vs FM 对比 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 各方法的 λ 估计值 ---
ax1 = axes[0]
methods = ['GMM\n(W=I)', 'GMM\n(W=S^-1)', 'FM\n(intercept)', 'FM\n(no intercept)']
lambdas = [lambda_1st, lambda_2nd, lambda_fm, lambda_fm_noint]
ses = [se_lambda_1st, se_lambda_2nd, se_fm, se_fm_noint]

x_pos = np.arange(len(methods))
bars = ax1.bar(x_pos, lambdas, yerr=[1.96*s for s in ses],
               color=['steelblue', '#2ecc71', '#e67e22', '#9b59b6'],
               edgecolor='black', alpha=0.8, capsize=5)

ax1.axhline(y=lambda_true, color='red', linestyle='--', linewidth=2,
            label=f'True $\\lambda$ = {lambda_true}')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(methods, fontsize=10)
ax1.set_ylabel('$\\hat{\\lambda}$ (%/month)', fontsize=12)
ax1.set_title('Factor Risk Premium Estimates', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# 标注值
for bar, v in zip(bars, lambdas):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# --- 右图: FM γ₁ 时间序列 ---
ax2 = axes[1]
ax2.bar(range(1, T_MONTHS+1), gamma_1_ts,
        color=['#2ecc71' if g > 0 else '#e74c3c' for g in gamma_1_ts],
        alpha=0.7, edgecolor='none')
ax2.axhline(y=lambda_fm, color='blue', linestyle='--', linewidth=2,
            label=f'Mean $\\gamma_1$ = {lambda_fm:.4f}')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('$\\gamma_{1,t}$ (%)', fontsize=12)
ax2.set_title('Fama-MacBeth: Monthly $\\gamma_1$ Estimates', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：各方法估计的 λ 值和 95% 置信区间（误差棒）")
print(f"       红色虚线 = 真实值 {lambda_true}")
print(f"       所有方法都能较好地估计 λ，但标准误有差异")
print(f"  右图：Fama-MacBeth 回归中每月的 γ₁ 估计值")
print(f"       正负交替但均值为正，说明因子风险溢价在统计上为正")

In [ ]:
# ========== 完整总结报告 ==========
print("=" * 60)
print("📋 GMM 因子模型检验完整报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   单因子模型能否解释 {N_ASSETS} 个资产的截面收益差异？")

print(f"\n📊 数据概况:")
print(f"   资产数量: {N_ASSETS}")
print(f"   样本期间: {T_MONTHS} 个月")
print(f"   因子个数: {K_FACTORS}")

print(f"\n🧮 参数估计:")
print(f"   因子风险溢价 λ̂ (两阶段 GMM) = {lambda_2nd:.4f}% / 月")
print(f"   标准误 SE(λ̂) = {se_lambda_2nd:.4f}")
print(f"   t 统计量 = {lambda_2nd/se_lambda_2nd:.4f}")
t_pval = 2 * (1 - stats.t.cdf(abs(lambda_2nd/se_lambda_2nd), df=T_MONTHS-1))
print(f"   P 值 = {t_pval:.6f}")

print(f"\n🧮 模型检验 (J 检验):")
print(f"   J 统计量 = {J_factor:.4f}")
print(f"   自由度 = {df_factor}")
print(f"   P 值 = {p_factor:.6f}")
print(f"   χ²({df_factor}) 临界值 (95%) = {chi2_crit:.4f}")

print(f"\n🎯 结论:")
if t_pval < 0.05:
    print(f"  ✓ 因子风险溢价 λ 显著不为零 (t = {lambda_2nd/se_lambda_2nd:.2f}, P = {t_pval:.4f})")
else:
    print(f"  ✗ 因子风险溢价 λ 不显著 (t = {lambda_2nd/se_lambda_2nd:.2f}, P = {t_pval:.4f})")

if p_factor > 0.05:
    print(f"  ✓ J 检验通过: 模型定价误差不显著 (J = {J_factor:.2f}, P = {p_factor:.4f})")
    print(f"  ✓ 结论: 单因子模型是可接受的")
else:
    print(f"  ✗ J 检验不通过: 定价误差联合显著 (J = {J_factor:.2f}, P = {p_factor:.4f})")
    print(f"  ✗ 结论: 单因子模型不充分，需要更多因子")

print("\n" + "=" * 60)

---

## 8. 核心概念回顾

### 📌 矩条件 (Moment Conditions)
- **定义**: 参数为真时期望为零的函数 $E[g(X, \theta)] = 0$
- **公式**: 样本类比 $\bar{g}_T(\theta) = \frac{1}{T}\sum_{t=1}^{T} g(X_t, \theta)$
- **含义**: 将"估计问题"转化为"让某些表达式的均值尽可能接近零"

### 📌 GMM 目标函数
- **定义**: 样本矩的加权二次型
- **公式**: $J(\theta) = \bar{g}_T(\theta)' W \bar{g}_T(\theta)$
- **含义**: 找到使所有矩条件的加权违反程度最小的参数值

### 📌 权重矩阵 (Weighting Matrix)
- **一阶段**: $W = I$（等权，简单但效率较低）
- **两阶段**: $W = \hat{S}^{-1}$（最优，效率最高）
- **含义**: 方差大的矩条件给低权重，方差小的给高权重

### 📌 J 检验 (Hansen's Test)
- **定义**: 过度识别约束的检验
- **公式**: $J = T \cdot \bar{g}_T(\hat{\theta})' \hat{S}^{-1} \bar{g}_T(\hat{\theta}) \sim \chi^2(m - k)$
- **判断标准**: P > 0.05 → 不拒绝模型; P < 0.05 → 模型可能误设

### 📌 GMM 在因子模型中的应用
- **矩条件**: $E[R_i^e - \beta_i' \lambda] = 0$（每个资产一个）
- **估计目标**: 因子风险溢价 $\lambda$
- **模型检验**: J 检验检验定价误差 $\alpha$ 是否联合为零

### 🔗 完整流程
```
指定矩条件 → 选择权重矩阵 → 最小化目标函数 → 获得参数估计
     ↓              ↓              ↓              ↓
   E[g]=0        W=I 或 Ŝ⁻¹     min ḡ'Wḡ      θ̂, SE(θ̂)
                                                  ↓
                                          过度识别? → J 检验
```

### 📝 检验指标汇总

| 概念 | 要点 | 判断标准 |
|------|------|----------|
| 矩条件 | $E[g(X, \theta)] = 0$ | 参数为真时期望为零 |
| 一阶段 GMM | $W = I$，简单 | 一致但非最优 |
| 两阶段 GMM | $W = \hat{S}^{-1}$，最优 | 渐近有效 |
| J 检验 | $J \sim \chi^2(m-k)$ | P > 0.05 → 模型可接受 |
| 识别条件 | $m \geq k$ | $m > k$ 时可做 J 检验 |

---

## 9. GMM 不应成为黑箱

### 📐 业务最优 vs 统计最优

从统计学角度，$W = S^{-1}$ 是最优权重矩阵，能给出最有效的估计量。

但从**业务角度**出发，选择权重矩阵 $a$ 时应当融入**领域知识**：

- 统计最优：$W = S^{-1}$（最小化渐近方差）
- 业务最优：根据经济理论选择 $a$（让最有经济意义的矩条件优先满足）

### 📖 资产定价中的经典例子

以 **CCAPM（消费资本资产定价模型）** 为例：

- 4 个资产：市场组合 (Market)、无风险利率 (Rf)、HML、SMB
- 2 个参数需要估计
- **业务最优做法**：选择 $a$ 让 Market 和 Rf 的矩条件**精确满足**（定价误差为零），用 HML 和 SMB 做模型检验（J 检验，df=2）
- **经济直觉**：CCAPM 应该能解释市场和无风险利率，如果连这两个都解释不了，模型就毫无意义

### 🎯 对比实验

下面我们用模拟数据对比两种策略：
- **业务最优 $a$**：根据经济含义选择矩条件
- **统计最优 $a$**：盲目使用 $W = S^{-1}$

In [ ]:
# ========== GMM 不应成为黑箱：业务最优 vs 统计最优 ==========
np.random.seed(42)

# --- 模拟 CCAPM 场景 ---
N_ASSET_BS = 4   # 4 个资产: Market, Rf, HML, SMB
T_BS = 120        # 120 个月
K_BS = 2          # 2 个因子

# 真实因子风险溢价
lambda_true_bs = np.array([0.5, 0.3])  # 市场溢价 + 规模溢价

# 各资产的因子暴露 (β 矩阵)
B_true = np.array([
    [1.0, 0.2],   # Market: 高市场暴露，低规模暴露
    [0.0, 0.0],   # Rf:    无暴露
    [0.3, 0.8],   # HML:   中等市场暴露，高价值暴露
    [0.2, 0.9],   # SMB:   低市场暴露，高规模暴露
])

# 生成收益数据
factor_rets = np.random.normal(lambda_true_bs, [4.0, 3.0], (T_BS, K_BS))
sigma_idio = 2.0
excess_rets_bs = factor_rets @ B_true.T + np.random.normal(0, sigma_idio, (T_BS, N_ASSET_BS))

R_bar_bs = excess_rets_bs.mean(axis=0)  # 各资产平均收益

print("📋 模拟 CCAPM 场景:")
print(f"  资产: Market, Rf, HML, SMB")
print(f"  参数: λ = {lambda_true_bs} (市场溢价, 规模溢价)")
print(f"  各资产平均超额收益: {np.round(R_bar_bs, 4)}")

# --- 策略 1: 业务最优 ---
# 让 Market 和 HML 的定价误差精确为零
# α = R̄ - B*λ = 0  对 Market 和 Rf
# 解: λ = (B_m'Rf')⁻¹ R̄_m'Rf'  (恰好识别)
B_select = B_true[[0, 2], :]  # Market 和 HML  # Market 和 Rf 的 β
R_select = R_bar_bs[[0, 2]]
lambda_biz = np.linalg.solve(B_select, R_select)
alpha_biz = R_bar_bs - B_true @ lambda_biz

print(f"\n📊 策略 1: 业务最优 (Market + HML 恰好识别)")
print(f"  λ̂_biz = {np.round(lambda_biz, 4)}")
print(f"  Market 定价误差: {alpha_biz[0]:.6f} (应为 0)")
print(f"  HML 定价误差:     {alpha_biz[3]:.6f} (应为 0)")
print(f"  HML 定价误差:    {alpha_biz[3]:.4f}")
print(f"  SMB 定价误差:    {alpha_biz[3]:.4f}")

# --- 策略 2: 统计最优 (W = S⁻¹) ---
# 计算 S 矩阵
pricing_err_bs = excess_rets_bs - np.outer(np.ones(T_BS), B_true @ lambda_biz)
S_bs = pricing_err_bs.T @ pricing_err_bs / T_BS
W_bs = np.linalg.inv(S_bs)

# GMM 优化
def gmm_obj_bs(lam, R_bar, B, W):
    alpha = R_bar - B @ lam
    return alpha @ W @ alpha

from scipy.optimize import minimize
res_stat = minimize(gmm_obj_bs, x0=[0, 0], args=(R_bar_bs, B_true, W_bs), method='Nelder-Mead')
lambda_stat = res_stat.x
alpha_stat = R_bar_bs - B_true @ lambda_stat

print(f"\n📊 策略 2: 统计最优 (W = S⁻¹)")
print(f"  λ̂_stat = {np.round(lambda_stat, 4)}")
print(f"  Market 定价误差: {alpha_stat[0]:.4f}")
print(f"  HML 定价误差:     {alpha_stat[2]:.4f}")
print(f"  HML 定价误差:    {alpha_stat[2]:.4f}")
print(f"  SMB 定价误差:    {alpha_stat[3]:.4f}")

In [ ]:
# ========== 可视化：业务最优 vs 统计最优 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

asset_names = ['Market', 'Rf', 'HML', 'SMB']
x_pos = np.arange(len(asset_names))

# --- 左图: 定价误差对比 ---
ax1 = axes[0]
width = 0.35
bars1 = ax1.bar(x_pos - width/2, alpha_biz, width, label='业务最优',
                color='steelblue', alpha=0.8, edgecolor='black')
bars2 = ax1.bar(x_pos + width/2, alpha_stat, width, label='统计最优',
                color='#e74c3c', alpha=0.8, edgecolor='black')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(asset_names)
ax1.set_ylabel('定价误差 α (%)', fontsize=12)
ax1.set_title('定价误差对比：业务最优 vs 统计最优', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 右图: λ 估计值对比 ---
ax2 = axes[1]
factor_names = ['市场溢价', '规模溢价']
x_f = np.arange(len(factor_names))
width2 = 0.25

ax2.bar(x_f - width2, lambda_true_bs, width2, label='真实值',
        color='#2ecc71', alpha=0.8, edgecolor='black')
ax2.bar(x_f, lambda_biz, width2, label='业务最优',
        color='steelblue', alpha=0.8, edgecolor='black')
ax2.bar(x_f + width2, lambda_stat, width2, label='统计最优',
        color='#e74c3c', alpha=0.8, edgecolor='black')

ax2.set_xticks(x_f)
ax2.set_xticklabels(factor_names)
ax2.set_ylabel('λ (%/月)', fontsize=12)
ax2.set_title('因子风险溢价估计对比', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 关键直觉:")
print(f"  1. 业务最优策略让 Market 和 HML 的定价误差精确为零")
print(f"     这符合经济学逻辑：CCAPM 应该能解释市场和无风险利率")
print(f"  2. 统计最优策略在所有资产上分配误差，可能让 Market 和 Rf 也有误差")
print(f"     统计上高效，但经济学上不合理")
print(f"  3. GMM 的灵活性是优势，但不要让它变成黑箱")
print(f"     应当从经济学出发，而非从统计学出发")
print(f"     选择最有经济含义的资产/矩条件来构建估计")

---

## 10. 常见误区

### ❌ 误区 1: J 检验不拒绝就说明模型"正确"
**✓ 正确理解**: J 检验是"不拒绝"而非"证明正确"。多个完全不同的模型可能都通过 J 检验。J 检验的功效 (power) 有限，尤其在小样本中可能无法检测出模型误设。

### ❌ 误区 2: 恰好识别时也可以做 J 检验
**✓ 正确理解**: 当矩条件数 $m$ 等于参数数 $k$ 时，矩条件可以精确满足，$J = 0$。J 检验只在过度识别 ($m > k$) 时才有意义。恰好识别的模型永远"通过"J 检验，但这并不说明模型正确。

### ❌ 误区 3: 矩条件越多越好
**✓ 正确理解**: 虽然更多矩条件理论上能提高估计效率，但在有限样本中，过多的矩条件会导致权重矩阵 $\hat{S}$ 的估计不精确（尤其当 $m$ 接近 $T$ 时），反而降低估计质量。这被称为"too many instruments"问题。

### ❌ 误区 4: 两阶段 GMM 一定优于一阶段
**✓ 正确理解**: 两阶段 GMM 在渐近意义上最优（$T \to \infty$），但在有限样本中，由于需要估计 $\hat{S}$，两阶段 GMM 可能存在更大的有限样本偏差。实践中，一些研究者偏好使用一阶段 GMM 的稳健标准误。

### ❌ 误区 5: GMM 不需要关心 β 的估计误差
**✓ 正确理解**: 在因子模型的 GMM 估计中，$\beta$ 是从第一步时间序列回归估计得到的（"生成的回归量"）。这引入了额外的估计误差，标准的 GMM 标准误可能低估真实的不确定性。Shanken (1992) 修正或使用包含第一步回归的联合 GMM 系统可以解决这个问题。